In [7]:
# ============================================================
# 📦 IMPORTS
# ============================================================

import ee
import pandas as pd
import io
from google.colab import files

# ============================================================
# 🔐 INITIALISATION GEE (AVEC PROJECT ID)
# ============================================================

PROJECT_ID = 'earth-engine-project-489915'

try:
    ee.Initialize(project=PROJECT_ID)
except Exception as e:
    print("🔐 Authentification requise...")
    ee.Authenticate()
    ee.Initialize(project=PROJECT_ID)

print(f"✅ GEE initialisé avec projet : {PROJECT_ID}")

# ============================================================
# ⚙️ PARAMÈTRES
# ============================================================

ANNEE_FIN = 2025
SCALE_M = 10000

# Define all relevant ERA5 bands for consistent processing
ERA5_BANDS = [
    'temperature_2m',
    'dewpoint_temperature_2m',
    'surface_pressure',
    'u_component_of_wind_10m',
    'v_component_of_wind_10m'
]

# ============================================================
# 📂 UPLOAD CSV
# ============================================================

print("📂 Téléchargez votre fichier stations_summary.csv :")
uploaded = files.upload()

if not uploaded:
    raise FileNotFoundError("❌ Aucun fichier téléchargé.")

filename = list(uploaded.keys())[0]
df_stations = pd.read_csv(io.StringIO(uploaded[filename].decode('utf-8')))

print("\n✅ Fichier chargé :")
display(df_stations.head())

# ============================================================
# 🚀 FONCTION EXTRACTION OPTIMISÉE + °C
# ============================================================

def extraire_quotidien_fast(region_id, lat, lon, annee_debut):
    point = ee.Geometry.Point([lon, lat])

    start = ee.Date(f"{int(annee_debut)}-01-01")
    end   = ee.Date(f"{ANNEE_FIN}-12-31")

    era5 = ee.ImageCollection('ECMWF/ERA5_LAND/HOURLY') \
        .filterDate(start, end) \
        .select(ERA5_BANDS) # Select all required bands upfront

    chirps = ee.ImageCollection('UCSB-CHG/CHIRPS/DAILY') \
        .filterDate(start, end)

    n_days = end.difference(start, 'day')
    days = ee.List.sequence(0, n_days.subtract(1))

    def compute_day(d):
        d = ee.Number(d)
        date = start.advance(d, 'day')
        next_date = date.advance(1, 'day')

        era5_day_collection = era5.filterDate(date, next_date) # Get hourly images for the day

        # 🔥 CONVERSION KELVIN → CELSIUS (SEULEMENT TEMP)
        # Map over the collection to apply conversion to each image, while keeping other bands
        def convert_temperatures_and_retain_bands(image):
            # Convert only temperature bands to Celsius
            temp_celsius = image.select('temperature_2m').subtract(273.15).rename('temperature_2m')
            dewpoint_celsius = image.select('dewpoint_temperature_2m').subtract(273.15).rename('dewpoint_temperature_2m')

            # Select other bands without conversion
            other_bands = image.select([
                'surface_pressure',
                'u_component_of_wind_10m',
                'v_component_of_wind_10m'
            ])

            # Merge all bands into a single image, ensuring consistency
            return temp_celsius.addBands(dewpoint_celsius).addBands(other_bands).copyProperties(image, image.propertyNames())

        era5_all_converted = era5_day_collection.map(convert_temperatures_and_retain_bands)

        # Stats - now operate on the converted collection
        era5_mean = era5_all_converted.mean() # Mean of all bands, with temp in Celsius
        temp_min_image  = era5_all_converted.select('temperature_2m').min() # Min of temperature_2m in Celsius
        temp_max_image  = era5_all_converted.select('temperature_2m').max() # Max of temperature_2m in Celsius

        # ====================================================
        # 📊 EXTRACTION
        # ====================================================

        values = era5_mean.reduceRegion(
            reducer=ee.Reducer.mean(),
            geometry=point,
            scale=SCALE_M,
            bestEffort=True
        )

        chirps_val = chirps.filterDate(date, next_date) \
            .sum().reduceRegion(
                reducer=ee.Reducer.mean(),
                geometry=point,
                scale=SCALE_M,
                bestEffort=True
        )

        tmin = temp_min_image.reduceRegion(
            ee.Reducer.min(), point, SCALE_M
        ).get('temperature_2m')

        tmax = temp_max_image.reduceRegion(
            ee.Reducer.max(), point, SCALE_M
        ).get('temperature_2m')

        return ee.Feature(None, values
            .combine(chirps_val)
            .set('date', date.format('YYYY-MM-dd'))
            .set('region_id', region_id)
            .set('temp_min', tmin)
            .set('temp_max', tmax)
        )

    fc = ee.FeatureCollection(days.map(compute_day))

    print(f"🚀 Extraction : {region_id} ({annee_debut} → {ANNEE_FIN})")

    data = fc.getInfo()

    rows = [f['properties'] for f in data['features']]
    df = pd.DataFrame(rows)

    df['datetime'] = pd.to_datetime(df['date'])
    df = df.sort_values('datetime')
    df.drop(columns=['date'], inplace=True)

    return df

# ============================================================
# 🔁 BOUCLE SUR LES STATIONS
# ============================================================

dfs = []

for _, row in df_stations.iterrows():
    try:
        df_r = extraire_quotidien_fast(
            region_id=row['nom'],
            lat=row['lat'],
            lon=row['lon'],
            annee_debut=row['depuis']
        )
        dfs.append(df_r)
    except Exception as e:
        print(f"❌ Erreur {row['nom']} : {e}")

# ============================================================
# 📊 CONCAT FINAL
# ============================================================

df_final = pd.concat(dfs, ignore_index=True)

print("\n✅ Extraction globale terminée !")
print(df_final.shape)

df_final.head()

✅ GEE initialisé avec projet : earth-engine-project-489915
📂 Téléchargez votre fichier stations_summary.csv :


Saving stations_summary.csv to stations_summary (2).csv

✅ Fichier chargé :


,nom,lat,lon,depuis
0,ksar El Kebir,35.003531,-5.915537,2012
1,DAR ELGUEDDARI,34.411281,-6.110075,2012
2,SIDI ALLAL TAZI,34.521974,-6.342802,2015
3,SUTA - FERME ABT,32.212324,-6.787755,2015
4,SUTA-TAZEROUALT,32.216766,-6.797247,2016


🚀 Extraction : ksar El Kebir (2012 → 2025)
❌ Erreur ksar El Kebir : Collection query aborted after accumulating over 5000 elements.
🚀 Extraction : DAR ELGUEDDARI (2012 → 2025)
❌ Erreur DAR ELGUEDDARI : Collection query aborted after accumulating over 5000 elements.
🚀 Extraction : SIDI ALLAL TAZI (2015 → 2025)
🚀 Extraction : SUTA - FERME ABT (2015 → 2025)
🚀 Extraction : SUTA-TAZEROUALT (2016 → 2025)
🚀 Extraction : SUTA OULAD AYAD (2015 → 2025)
🚀 Extraction : MECHRAA BELKSIRI (2018 → 2025)
🚀 Extraction : SUTA 507 (2019 → 2025)
🚀 Extraction : M-Zaio (2016 → 2025)
🚀 Extraction : Merksen (2017 → 2025)
🚀 Extraction : SUTA_CENTAGRI (2017 → 2025)
🚀 Extraction : SUTA INRA (2018 → 2025)
🚀 Extraction : SUTA 503 (2018 → 2025)
🚀 Extraction : SUTA 505 (2018 → 2025)
🚀 Extraction : SUTA Ait Ali (2018 → 2025)
🚀 Extraction : SUTA 522 (2018 → 2025)
🚀 Extraction : M-Berkane (2021 → 2025)
🚀 Extraction : M-Nador (2019 → 2025)
🚀 Extraction : DK-GHARBIA 343 (2019 → 2025)
🚀 Extraction : DK-SEMVAD (2019 → 2025)

,dewpoint_temperature_2m,precipitation,region_id,surface_pressure,temp_max,temp_min,temperature_2m,u_component_of_wind_10m,v_component_of_wind_10m,datetime
0,7.052469,0.000000,SIDI ALLAL TAZI,102936.548991,16.947443,5.472513,10.602466,-0.705336,-0.462993,2015-01-01
1,5.350747,0.000000,SIDI ALLAL TAZI,103166.083984,17.241785,5.772150,10.940889,-1.576534,-0.576786,2015-01-02
2,6.380649,10.966801,SIDI ALLAL TAZI,103166.289062,18.683862,6.555505,11.524372,-1.319960,-0.680841,2015-01-03
3,8.354894,0.000000,SIDI ALLAL TAZI,103041.631022,17.432016,6.409647,11.147799,0.022695,0.519074,2015-01-04
4,9.088668,0.000000,SIDI ALLAL TAZI,102805.420085,16.657480,6.946695,11.089300,0.038571,0.002300,2015-01-05


In [8]:
from google.colab import sheets
sheet = sheets.InteractiveSheet(df=df_final)

https://docs.google.com/spreadsheets/d/1cjVlFxy1C9AQ82jXH1f6JjNK0tkvkEXWGxnsMX7XW7U/edit#gid=0
